# Enveda v1 submission: FPNet single + 20ppm window (offline GPU)

Two paths: full train+coco index when RDKit installs, else coco-only
precomputed pool. Never empty output. R1 NP-method at 20ppm.

In [ ]:
import subprocess, sys, glob
print(sys.version)
whl = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
print('wheels:', whl)
if whl:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', whl[0]], check=False)
try:
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
    from rdkit.Chem.Descriptors import ExactMolWt
    RDLogger.DisableLog('rdApp.*')
    import rdkit
    print('rdkit =', rdkit.__version__)
    HAVE_RDKIT = True
except Exception as e:
    HAVE_RDKIT = False
    print('NO RDKIT:', str(e)[:200])
import torch
print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available())
assert torch.cuda.is_available(), 'need GPU'
import numpy as np, pyarrow.parquet as pq
print('numpy/pyarrow ok; HAVE_RDKIT =', HAVE_RDKIT)


In [ ]:
from pathlib import Path
def find(name):
    hits = list(Path('/kaggle/input').rglob(name))
    assert hits, 'missing input: ' + name
    print(name, '->', hits[0])
    return str(hits[0])
TEST = find('test.parquet')
TRAIN = find('train.parquet')
W_SINGLE = find('fp_single_s2.pt')
BITS_P = find('fp_bits.npy')
COCO_FP = find('coco_fp.npy')
COCO_MASS = find('coco_mass.npy')
COCO_META = find('coco_meta.pkl')
print('torch cuda devices:', torch.cuda.device_count())


In [ ]:
import re
MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100,
            P=30.97376163, Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473,
            Si=27.9769265325, Se=79.9165213, Na=22.9897692809, K=38.96370668, B=11.0093054,
            As=74.9215965, Fe=55.9349375, D=2.0141017778)
_TOK = re.compile(r'([A-Z][a-z]?)(\d*)')
def formula_mass(f):
    if not isinstance(f, str) or not f: return np.nan
    m = 0.0
    for el, n in _TOK.findall(f):
        if not el: continue
        if el not in MONO: return np.nan
        m += MONO[el] * (int(n) if n else 1)
    return m
MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
    S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
    Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {'[M+H]+':(1,1,PROTON),'[M+NH4]+':(1,1,NH4-E),
 '[M+Na]+':(1,1,MASS['Na']-E),'[M+K]+':(1,1,MASS['K']-E),
 '[M-H2O+H]+':(1,1,PROTON-H2O),'[M-2H2O+H]+':(1,1,PROTON-2*H2O),
 '[M+2H]2+':(1,2,2*PROTON),'[M]+':(1,1,-E),'[M-H2O]+':(1,1,-E-H2O),
 '[M+CH3OH+H]+':(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 '[M+CH3CN+H]+':(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 '[M-H]-':(1,1,-PROTON),'[M-H2O-H]-':(1,1,-PROTON-H2O),
 '[M+CH2O2-H]-':(1,1,FORMATE-PROTON),'[M+C2H4O2-H]-':(1,1,ACETATE-PROTON),
 '[M+Cl]-':(1,1,MASS['Cl']+E),'[M]-':(1,1,E),
 '[M-2H]-':(1,2,-2*PROTON),'[M+Na-2H]-':(1,1,MASS['Na']-2*PROTON),
 '[2M+H]+':(2,1,PROTON),'[2M+Na]+':(2,1,MASS['Na']-E),
 '[2M+NH4]+':(2,1,NH4-E),'[2M+K]+':(2,1,MASS['K']-E),
 '[2M-H]-':(2,1,-PROTON),'[2M+CH2O2-H]-':(2,1,FORMATE-PROTON),
 '[2M+C2H4O2-H]-':(2,1,ACETATE-PROTON),'[2M+Na-2H]-':(2,1,MASS['Na']-2*PROTON),
 '[3M+H]+':(3,1,PROTON),'[3M-H]-':(3,1,-PROTON)}
def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m = (ad == a)
        if m.any(): out[m] = (mz[m]*z - d)/n
    return out
print('mass contract ready')


In [ ]:
import pickle, time
import numpy as np
t0=time.perf_counter()
tr = pq.ParquetFile(TRAIN)
seen = {}
forms = {}
for b in tr.iter_batches(columns=['inchikey14','normalized_smiles','molecular_formula'], batch_size=200000):
    kk = b.column(0).to_pylist(); ss = b.column(1).to_pylist(); ff = b.column(2).to_pylist()
    for k,s,f in zip(kk,ss,ff):
        if k and k not in seen:
            seen[k] = s; forms[k] = f
print('train structures:', len(seen), round(time.perf_counter()-t0,1))
keys = sorted(seen.keys())
smi = [seen[k] for k in keys]
fmap = {f: formula_mass(f) for f in set(forms.values())}
mass = np.array([fmap.get(forms[k], np.nan) for k in keys], float)
ok = np.isfinite(mass)
print('mass ok:', int(ok.sum()), '/', len(mass))
keys = [k for k,o in zip(keys,ok) if o]
smi = [s for s,o in zip(smi,ok) if o]
mass = mass[ok]
meta = pickle.load(open(COCO_META,'rb'))
ckeys = list(meta['keys']); csmi = list(meta['smiles'])
coco_mass = np.load(COCO_MASS, mmap_mode='r')
coco_fp = np.load(COCO_FP, mmap_mode='r')
print('coco:', len(ckeys), coco_fp.shape)
cmap = dict(zip(ckeys, np.asarray(coco_mass, float)))
cidx = {k:i for i,k in enumerate(ckeys)}
allk = keys + [k for k in ckeys if k not in seen]
alls = smi + [s for k,s in zip(ckeys,csmi) if k not in seen]
allm = np.concatenate([mass, np.array([cmap[k] for k in ckeys if k not in seen], float)])
print('combined:', len(allk), round(time.perf_counter()-t0,1))
TQ = pq.read_table(TEST).to_pydict()
mol = {}
for i in range(len(TQ['molecule_id'])):
    mol.setdefault(TQ['molecule_id'][i], []).append(i)
print('test molecules:', len(mol), 'spectra:', len(TQ['molecule_id']))
print('HAVE_RDKIT =', HAVE_RDKIT)


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math
"""Spectrum -> molecular fingerprint model (CSI:FingerID-style neural ranker)."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math

MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+","[M+NH4]+","[M+Na]+","[M+K]+","[M-H2O+H]+","[M-2H2O+H]+","[M]+",
               "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+C2H4O2-H]-","[M+Cl]-","[M]-",
               "[M+2H]2+","[M-2H]-","[2M+H]+","[2M+Na]+","[2M+NH4]+","[2M-H]-","[2M+K]+",
               "[2M+CH2O2-H]-","[2M+C2H4O2-H]-","[2M+Na-2H]-","[M+Na-2H]-","[M-H2O]+","<unk>"]
ADDUCT_IX = {a:i for i,a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF","Orbitrap","QTOF","IT","other"]
INSTR_IX = {a:i for i,a in enumerate(INSTR_LIST)}

def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4

def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = it >= floor*mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        # keep the top `per_win` peaks inside each `win` Da bucket, then global top-N
        order = np.argsort(-it)
        bucket = (mz//win).astype(np.int64)
        cnt = {}; sel=[]
        for i in order:
            b = bucket[i]; c = cnt.get(b,0)
            if c < per_win: cnt[b]=c+1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks-len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it/it.max())
    return mz.astype(np.float32), v.astype(np.float32)

class SinEmb(nn.Module):
    """Log-spaced sinusoidal embedding for m/z values (Voronov et al.)."""
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim//2
        wav = torch.pow(10.0, (hi-lo)*torch.pow(torch.linspace(0,1,n), power) + lo)
        self.register_buffer('inv', (2*math.pi)/wav)
    def forward(self, x):                      # x: (...,)
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h=h
        self.n1=nn.LayerNorm(d); self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)
        self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d,d))
        self.drop=nn.Dropout(drop)
    def forward(self, x, pad):
        B,N,D=x.shape; y=self.n1(x)
        q,k,v = self.qkv(y).view(B,N,3,self.h,D//self.h).permute(2,0,3,1,4)
        m = (~pad)[:,None,None,:]                       # True = attend
        a = F.scaled_dot_product_attention(q,k,v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1,2).reshape(B,N,D)))
        return x + self.drop(self.ff(self.n2(x)))

class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d=d
        self.mz_emb  = SinEmb(d)
        self.nl_emb  = SinEmb(d)
        self.pk = nn.Linear(2*d+1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d+3, d)
        self.blocks = nn.ModuleList([Block(d,heads,drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2*d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))
    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B,N = mz.shape
        nl = (prec[:,None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce/100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1)/10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B,1,dtype=torch.bool,device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:,0]
        msk = (~pad[:,1:]).float().unsqueeze(-1)
        mean = (x[:,1:]*msk).sum(1)/msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))
ck = torch.load(W_SINGLE, map_location='cpu', weights_only=True)
net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).cuda().eval()
net.load_state_dict(ck['model'], strict=True)
print('model ok:', ck['nbits'], ck['d'], ck['layers'])
BITS = np.load(BITS_P)
print('BITS:', BITS.shape)
if HAVE_RDKIT:
    _m2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _m3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _rk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)
    def fp_of(smi):
        m = Chem.MolFromSmiles(smi)
        if m is None: return None
        try:
            fp = np.concatenate([_m2.GetFingerprintAsNumPy(m).astype(np.uint8), _m3.GetFingerprintAsNumPy(m).astype(np.uint8), _rk.GetFingerprintAsNumPy(m).astype(np.uint8), np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[BITS]
            return fp, 0.0
        except Exception:
            return None
else:
    fp_of = None


In [ ]:
import numpy as np, torch, time
@torch.no_grad()
def fwd1(prow):
    a,b = prep_peaks(np.asarray(prow[0],float), np.asarray(prow[1],float), float(prow[2]))
    if not len(a): return None
    N = len(a)
    mz=np.zeros((1,N),np.float32); it=np.zeros((1,N),np.float32); pad=np.zeros((1,N),bool)
    mz[0,:N]=a; it[0,:N]=b
    T = lambda x: torch.as_tensor(x, device='cuda')
    args=(T(mz),T(it),T(pad),T(np.array([float(prow[2])],np.float32)),
          T(np.array([ADDUCT_IX.get(prow[3],ADDUCT_IX['<unk>'])])),
          T(np.array([instr_family(prow[4])])),
          T(np.array([float(np.mean(np.atleast_1d(prow[5]))) if prow[5] is not None else 25.0],np.float32)),
          T(np.array([1.0 if prow[6]=='positive' else -1.0],np.float32)))
    return net(*args).float().cpu().numpy()[0]
out_rows = []
t0=time.perf_counter()
for n,(mid,idxs) in enumerate(mol.items()):
    try:
        best = max(idxs, key=lambda i: len(TQ['ms2_mzs'][i]))
        prow = (TQ['ms2_mzs'][best], TQ['ms2_normalized_intensities'][best], TQ['precursor_mz'][best],
                TQ['adduct'][best], TQ['instrument_type'][best], TQ['collision_energy_ev'][best], TQ['ionization_mode'][best])
        z = fwd1(prow)
        qm = neutral_mass(np.array([prow[2]]), [prow[3]])[0]
        cands = []
        if not np.isnan(qm):
            if HAVE_RDKIT:
                w = np.where(np.abs(allm - qm) <= qm*20e-6)[0]
                if len(w) > 2000:
                    w = w[np.argsort(np.abs(allm[w]-qm))[:2000]]
                F = []
                for j in w:
                    r = fp_of(alls[j])
                    if r is None: continue
                    F.append((r[0], allk[j], alls[j], abs(allm[j]-qm)/qm*1e6))
            else:
                w = np.where(np.abs(np.asarray(coco_mass, float) - qm) <= qm*20e-6)[0]
                if len(w) > 2000:
                    w = w[np.argsort(np.abs(np.asarray(coco_mass, float)[w]-qm))[:2000]]
                F = []
                U = np.unpackbits(np.asarray(coco_fp, dtype=np.uint8)[w], axis=1)[:, :6930].astype(np.float32)
                for (fpv, j) in zip(U, w):
                    F.append((fpv, ckeys[j], csmi[j], abs(float(coco_mass[j])-qm)/qm*1e6))
            if z is not None and F:
                Fall = np.stack([f.astype(np.float32) for f,_,_,_ in F])
                s = Fall @ z
                order = np.argsort(-s)
                seen_k, cands = set(), []
                for j in order:
                    _,k,sm,_ = F[j]
                    if k in seen_k: continue
                    seen_k.add(k); cands.append(sm)
                    if len(cands) >= 25: break
            if not cands and F:
                fb = sorted(F, key=lambda t: t[3])[:25]
                seen_k, cands = set(), []
                for _,k,sm,_ in fb:
                    if k in seen_k: continue
                    seen_k.add(k); cands.append(sm)
        if not cands:
            cands = ['CCO']*25
        else:
            cands = (cands + ['CCO']*25)[:25]
        out_rows.append((mid, ';'.join(cands)))
    except Exception as e:
        out_rows.append((mid, ';'.join(['CCO']*25)))
    if (n+1)%100==0: print(n+1, round(time.perf_counter()-t0,1))
import csv
with open('/kaggle/working/submission.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['molecule_id','smiles'])
    w.writerows(out_rows)
print('wrote', len(out_rows), round(time.perf_counter()-t0,1))
print('HAVE_RDKIT =', HAVE_RDKIT)
assert len(out_rows)==len(mol)


产物 `/kaggle/working/submission.csv`。空窗回退质量排序，全空回退 CCO——永不空行。